# Task 1 - Club Signups Cleaning

This notebook profiles the raw sign-up data, standardizes inconsistent values, handles duplicates, and writes the cleaned CSV.

## Imports

In [1]:
import pandas as pd

## First Look / Profile Raw Data

Initial checks on the raw file before creating the cleaned working copy.

In [2]:
raw = pd.read_csv("club_signups.csv")
raw

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes
5,S012,55-9245,ALI IBRAHIM,ali.ibrahim@student.guc.edu.eg,MET,cairo,Robotics,15/09/2026 03:42,N
6,S007,61-2495,ahmed adel,AHMED.ADEL@STUDENT.GUC.EDU.EG,Pharma,Alex,Robotics,2026-09-14 17:03,Y
7,S005,61-4733,OMAR ADEL,omar.adel@student.guc.edu.eg,EMS,cairo,football,2026-09-13 23:52,1
8,S020,55-9245,ali ibrahim,ali.ibrahim@student.guc.edu.eg,MET,CAIRO,Music,2026-09-15 23:07,false
9,S033,55-9245,ali ibrahim,ali.ibrahim@student.guc.edu.eg,met,Cairo,music,2026-09-16 09:07,1


In [3]:
raw.shape

(39, 9)

In [4]:
raw.dtypes

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

In [5]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   signup_id     39 non-null     object
 1   student_id    39 non-null     object
 2   full_name     39 non-null     object
 3   email         39 non-null     object
 4   faculty       39 non-null     object
 5   city          39 non-null     object
 6   club          39 non-null     object
 7   signed_up_at  39 non-null     object
 8   fee_paid      39 non-null     object
dtypes: object(9)
memory usage: 2.9+ KB


In [6]:
raw.isna().sum()

signup_id       0
student_id      0
full_name       0
email           0
faculty         0
city            0
club            0
signed_up_at    0
fee_paid        0
dtype: int64

In [7]:
category_columns = ["faculty", "city", "club", "fee_paid"]

for column in category_columns:
    print(f"\n{column}")
    print(raw[column].value_counts(dropna=False))


faculty
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

city
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1
El Giza        1
Name: count, dtype: int64

club
club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club   

In [8]:
raw["signed_up_at"].agg(["min", "max"])

min    13/09/2026 15:38
max    2026-09-17 23:36
Name: signed_up_at, dtype: object

In [9]:
print("Rows:", len(raw))
print("Unique signup_id values:", raw["signup_id"].nunique())
print("Exact duplicate rows:", raw.duplicated().sum())
print("Duplicate signup_id rows:", raw.duplicated("signup_id").sum())

Rows: 39
Unique signup_id values: 36
Exact duplicate rows: 3
Duplicate signup_id rows: 3


## Copy Raw Data

Create a working dataframe so the original CSV load stays available for comparison.

In [10]:
signups = raw.copy()
signups.head()

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


## Fix Column Names

Use a consistent column naming style before cleaning the values.

In [11]:
signups.columns = (
    signups.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "_", regex=False)
)

signups.columns

Index(['signup_id', 'student_id', 'full_name', 'email', 'faculty', 'city',
       'club', 'signed_up_at', 'fee_paid'],
      dtype='object')

## Standardize Student Names and Emails

Clean spacing and capitalization in names, and standardize emails to lowercase.

In [12]:
signups["full_name"] = signups["full_name"].str.split().str.join(" ").str.title()
signups["email"] = signups["email"].str.strip().str.lower()

signups[["student_id", "full_name", "email"]].head(10)

,student_id,full_name,email
0,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg
1,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg
2,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg
3,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg
4,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg
5,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
6,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg
7,61-4733,Omar Adel,omar.adel@student.guc.edu.eg
8,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg
9,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg


In [13]:
signups[["full_name", "email"]].value_counts().head(10)

full_name       email                            
Mohamed Adel    mohamed.adel@student.guc.edu.eg      6
Laila Mostafa   laila.mostafa@student.guc.edu.eg     6
Ali Ibrahim     ali.ibrahim@student.guc.edu.eg       5
Ziad Saleh      ziad.saleh@student.guc.edu.eg        3
Sara Zaki       sara.zaki@student.guc.edu.eg         3
Hana Zaki       hana.zaki@student.guc.edu.eg         3
Youssef Fathy   youssef.fathy@student.guc.edu.eg     2
Karim Kamel     karim.kamel@student.guc.edu.eg       2
Ahmed Adel      ahmed.adel@student.guc.edu.eg        2
Farida Ibrahim  farida.ibrahim@student.guc.edu.eg    2
Name: count, dtype: int64

## Standardize Faculties, Clubs, and Fee Paid

Normalize text values, then map known variants to the canonical categories.

In [14]:
faculty_map = {
    "met": "MET",
    "media engineering and technology": "MET",
    "iet": "IET",
    "information engineering and technology": "IET",
    "ems": "EMS",
    "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy",
    "pharma": "Pharmacy",
    "management": "Management",
}

club_map = {
    "robotics": "Robotics",
    "debate": "Debate",
    "debating": "Debate",
    "debate club": "Debate",
    "music": "Music",
    "football": "Football",
    "soccer": "Football",
    "chess": "Chess",
    "chess club": "Chess",
}

city_map = {
    "cairo": "Cairo",
    "giza": "Giza",
    "el giza": "Giza",
    "alexandria": "Alexandria",
    "alex": "Alexandria",
}

fee_paid_map = {
    "yes": True,
    "y": True,
    "true": True,
    "1": True,
    "no": False,
    "n": False,
    "false": False,
    "0": False,
}

In [15]:
signups["faculty"] = signups["faculty"].str.strip().str.lower().map(faculty_map)
signups["city"] = signups["city"].str.strip().str.lower().map(city_map)
signups["club"] = signups["club"].str.strip().str.lower().map(club_map)
signups["fee_paid"] = signups["fee_paid"].astype(str).str.strip().str.lower().map(fee_paid_map)

signups[["faculty", "city", "club", "fee_paid"]].head(10)

,faculty,city,club,fee_paid
0,Pharmacy,Giza,Football,False
1,Management,Alexandria,Chess,False
2,EMS,Cairo,Music,True
3,EMS,Cairo,Debate,True
4,IET,Alexandria,Chess,True
5,MET,Cairo,Robotics,False
6,Pharmacy,Alexandria,Robotics,True
7,EMS,Cairo,Football,True
8,MET,Cairo,Music,False
9,MET,Cairo,Music,True


In [16]:
for column in ["faculty", "city", "club", "fee_paid"]:
    print(f"\n{column}")
    print(signups[column].value_counts(dropna=False))


faculty
faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

city
city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64

club
club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

fee_paid
fee_paid
True     27
False    12
Name: count, dtype: int64


In [17]:
allowed_faculties = {"MET", "IET", "EMS", "Pharmacy", "Management"}
allowed_cities = {"Cairo", "Giza", "Alexandria"}
allowed_clubs = {"Robotics", "Debate", "Music", "Football", "Chess"}

assert set(signups["faculty"].dropna()).issubset(allowed_faculties)
assert set(signups["city"].dropna()).issubset(allowed_cities)
assert set(signups["club"].dropna()).issubset(allowed_clubs)
assert signups["fee_paid"].dropna().isin([True, False]).all()
assert signups[["faculty", "city", "club", "fee_paid"]].isna().sum().sum() == 0

## Inspect Signup Dates

Check the date patterns present in `signed_up_at` before parsing.

In [18]:
signups["signed_up_at"].value_counts().sort_index()

signed_up_at
13/09/2026 15:38    1
14/09/2026 01:22    1
15/09/2026 03:12    1
15/09/2026 03:28    1
15/09/2026 03:42    1
15/09/2026 17:20    1
15/09/2026 18:01    1
15/09/2026 18:40    1
15/09/2026 20:20    1
16/09/2026 01:15    1
16/09/2026 02:01    1
16/09/2026 23:55    1
17/09/2026 12:27    1
18/09/2026 23:36    1
2026-09-13 14:36    1
2026-09-13 21:16    1
2026-09-13 22:36    1
2026-09-13 23:52    1
2026-09-14 17:03    1
2026-09-14 23:32    1
2026-09-15 01:05    1
2026-09-15 05:43    1
2026-09-15 11:36    1
2026-09-15 13:39    2
2026-09-15 23:07    1
2026-09-16 01:39    2
2026-09-16 09:07    1
2026-09-16 09:29    2
2026-09-16 16:16    1
2026-09-16 20:31    1
2026-09-17 00:41    1
2026-09-17 06:54    1
2026-09-17 11:15    1
2026-09-17 17:14    1
2026-09-17 23:29    1
2026-09-17 23:36    1
Name: count, dtype: int64

In [19]:
iso_format = signups["signed_up_at"].str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$")
slash_format = signups["signed_up_at"].str.match(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$")

print("ISO yyyy-mm-dd dates:", iso_format.sum())
print("Slash dd/mm/yyyy dates:", slash_format.sum())
print("Unexpected date formats:", (~(iso_format | slash_format)).sum())

ISO yyyy-mm-dd dates: 25
Slash dd/mm/yyyy dates: 14
Unexpected date formats: 0


In [20]:
signups.loc[~(iso_format | slash_format), "signed_up_at"]

Series([], Name: signed_up_at, dtype: object)

## Normalize Signup Dates

Parse both date formats into one datetime column, treating slash dates as day/month/year.

In [21]:
signups["signed_up_at"] = pd.to_datetime(
    signups["signed_up_at"],
    format="mixed",
    dayfirst=True,
)

signups[["signup_id", "student_id", "club", "signed_up_at"]].head(10)

,signup_id,student_id,club,signed_up_at
0,S015,61-8942,Football,2026-09-15 13:39:00
1,S010,61-3846,Chess,2026-09-15 03:12:00
2,S008,61-4844,Music,2026-09-14 23:32:00
3,S035,61-4844,Debate,2026-09-17 11:15:00
4,S004,61-7344,Chess,2026-09-13 22:36:00
5,S012,55-9245,Robotics,2026-09-15 03:42:00
6,S007,61-2495,Robotics,2026-09-14 17:03:00
7,S005,61-4733,Football,2026-09-13 23:52:00
8,S020,55-9245,Music,2026-09-15 23:07:00
9,S033,55-9245,Music,2026-09-16 09:07:00


In [22]:
print("dtype:", signups["signed_up_at"].dtype)
print("missing dates:", signups["signed_up_at"].isna().sum())
print("earliest signup:", signups["signed_up_at"].min())
print("latest signup:", signups["signed_up_at"].max())

dtype: datetime64[ns]
missing dates: 0
earliest signup: 2026-09-13 14:36:00
latest signup: 2026-09-18 23:36:00


In [23]:
assert pd.api.types.is_datetime64_any_dtype(signups["signed_up_at"])
assert signups["signed_up_at"].isna().sum() == 0

## Exact Duplicates

Rows that match on every column are duplicate copies of the same submission, so one copy is enough.

In [24]:
exact_duplicate_count = signups.duplicated().sum()
exact_duplicate_count

np.int64(3)

In [25]:
signups[signups.duplicated(keep=False)].sort_values(["signup_id", "student_id", "club", "signed_up_at"])

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [26]:
signups = signups.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(signups))
print("Remaining exact duplicates:", signups.duplicated().sum())

Rows after removing exact duplicates: 36
Remaining exact duplicates: 0


## Investigate Other Duplicates

After removing exact duplicates, look at repeated IDs, emails, and student-club pairs.

In [27]:
print("Duplicate signup_id rows:", signups.duplicated("signup_id").sum())
print("Duplicate student_id rows:", signups.duplicated("student_id").sum())
print("Duplicate email rows:", signups.duplicated("email").sum())
print("Duplicate student_id + club rows:", signups.duplicated(["student_id", "club"]).sum())

Duplicate signup_id rows: 0
Duplicate student_id rows: 21
Duplicate email rows: 22
Duplicate student_id + club rows: 4


A repeated student or email is fine if the clubs are different. The rows to remove are repeated `student_id` + `club` pairs.

In [28]:
student_summary = (
    signups.groupby(["student_id", "email", "full_name"])
    .agg(
        rows=("signup_id", "count"),
        different_clubs=("club", "nunique"),
        repeated_same_club=("club", lambda clubs: clubs.duplicated().sum()),
    )
    .query("rows > 1")
    .sort_values(["repeated_same_club", "rows"], ascending=False)
)

student_summary

,,,rows,different_clubs,repeated_same_club
student_id,email,full_name,,,
61-7344,laila.mostafa@student.guc.edu.eg,Laila Mostafa,6,4,2
55-9245,ali.ibrahim@student.guc.edu.eg,Ali Ibrahim,5,4,1
61-4844,mohamed.adel@student.guc.edu.eg,Mohamed Adel,4,3,1
55-2639,youssef.fathy@student.guc.edu.eg,Youssef Fathy,2,2,0
55-2992,mohamed.adel@student.guc.edu.eg,Mohamed Adel,2,2,0
58-1089,karim.kamel@student.guc.edu.eg,Karim Kamel,2,2,0
58-1125,ziad.saleh@student.guc.edu.eg,Ziad Saleh,2,2,0
58-8701,farida.ibrahim@student.guc.edu.eg,Farida Ibrahim,2,2,0
61-2495,ahmed.adel@student.guc.edu.eg,Ahmed Adel,2,2,0


In [29]:
signups[signups.duplicated("signup_id", keep=False)].sort_values(["signup_id", "signed_up_at"])

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid


In [30]:
student_club_duplicates = signups[signups.duplicated(["student_id", "club"], keep=False)]

student_club_duplicates.sort_values(["student_id", "club", "signed_up_at"])

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


In [31]:
repeated_students = signups[signups.duplicated("student_id", keep=False)]

repeated_students.sort_values(["student_id", "signed_up_at", "club"])

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
31,S014,55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg,Management,Cairo,Chess,2026-09-15 11:36:00,True
27,S016,55-2639,Youssef Fathy,youssef.fathy@student.guc.edu.eg,Management,Cairo,Robotics,2026-09-15 17:20:00,True
14,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
18,S024,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Robotics,2026-09-16 16:16:00,True
10,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
5,S012,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Robotics,2026-09-15 03:42:00,False
37,S013,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Football,2026-09-15 05:43:00,True
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
36,S028,58-1089,Karim Kamel,karim.kamel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-17 06:54:00,True


## Remove Repeated Same-Club Signups

For repeated `student_id` + `club` pairs, keep the latest row because it reflects the newest fee status.

In [32]:
rows_before_repeated_signup_removal = len(signups)
repeated_signup_count = signups.duplicated(["student_id", "club"]).sum()

print("Rows before removing repeated same-club signups:", rows_before_repeated_signup_removal)
print("Repeated same-club rows to remove:", repeated_signup_count)

Rows before removing repeated same-club signups: 36
Repeated same-club rows to remove: 4


In [33]:
repeated_signups = signups[signups.duplicated(["student_id", "club"], keep=False)]

repeated_signups.sort_values(["student_id", "club", "signed_up_at"])

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Football,2026-09-18 23:36:00,True


In [34]:
signups = (
    signups.sort_values("signed_up_at")
    .drop_duplicates(["student_id", "club"], keep="last")
    .sort_values("signup_id")
    .reset_index(drop=True)
)

rows_removed_repeated_signups = rows_before_repeated_signup_removal - len(signups)

print("Rows removed:", rows_removed_repeated_signups)
print("Rows remaining:", len(signups))

Rows removed: 4
Rows remaining: 32


In [35]:
assert rows_removed_repeated_signups == repeated_signup_count
assert signups.duplicated(["student_id", "club"]).sum() == 0

print("student_id + club is unique:", signups.duplicated(["student_id", "club"]).sum() == 0)

student_id + club is unique: True


In [36]:
signups

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


## Save Clean Data

Write the final cleaned dataset.

In [37]:
signups.to_csv("club_signups_clean.csv", index=False)

print("Saved club_signups_clean.csv")

Saved club_signups_clean.csv
